In [0]:
%pip install lakebench[tpcds_datagen]

In [0]:
%restart_python

In [0]:
dbutils.widgets.dropdown("SF", "10", ["10", "100", "1000"])
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("schema_name", "")

In [0]:
catalog_name=dbutils.widgets.get('catalog_name')
schema_name=dbutils.widgets.get('schema_name')
scale_factor=dbutils.widgets.get('SF')

In [0]:
from lakebench.datagen import TPCDSDataGenerator
import duckdb

volume_path = f'/Volumes/{catalog_name}/{schema_name}/datagen/tpcds_sf{scale_factor}'
temp_dir = f'/local_disk0/tmp/duckdb_tmp_sf{scale_factor}'   # set as needed depending on the compute size used to generate the data

_original_duckdb_connect = duckdb.connect

def _connect_with_temp_dir(*args, **kwargs):
    con = _original_duckdb_connect(*args, **kwargs)
    con.execute(f"SET temp_directory='{temp_dir}'") # set as needed depending on the compute size used to generate the data
    con.execute(f"SET memory_limit='400GB'")   # set as needed depending on the compute size used to generate the data
    return con

try:
    duckdb.connect = _connect_with_temp_dir
    datagen = TPCDSDataGenerator(
        scale_factor=scale_factor,
        target_folder_uri=volume_path
    )
    datagen.run()
finally:
    duckdb.connect = _original_duckdb_connect

In [0]:
tables = ['catalog_page', 'catalog_sales', 'customer_address', 'customer_demographics', 'date_dim', 'item', 'promotion', 'ship_mode', 'store', 'store_sales']

int_columns = [
    'd_date_sk', 'd_month_seq', 'd_week_seq', 'd_quarter_seq', 'd_year', 
    'd_dow', 'd_moy', 'd_dom', 'd_qoy', 'd_fy_year', 'd_fy_quarter_seq', 
    'd_fy_week_seq', 'd_first_dom', 'd_last_dom', 'd_same_day_ly', 'd_same_day_lq','cp_catalog_page_sk', 'cp_start_date_sk','cp_end_date_sk','cp_catalog_number','cp_catalog_page_number','ca_address_sk','cd_demo_sk','cd_purchase_estimate','cd_dep_count','cd_dep_employed_count','cd_dep_college_count','i_item_sk','i_brand_id','i_class_id','i_category_id','i_manufact_id','i_manager_id','p_promo_sk','p_start_date_sk','p_end_date_sk','p_item_sk','p_response_target','sm_ship_mode_sk','s_store_sk','s_closed_date_sk','s_number_employees','s_floor_space','s_market_id','s_division_id','s_company_id','cs_call_center_sk','cs_catalog_page_sk','cs_ship_mode_sk','cs_bill_customer_sk','cs_bill_cdemo_sk','cs_bill_hdemo_sk','cs_bill_addr_sk','cs_ship_customer_sk','cs_ship_cdemo_sk','cs_ship_hdemo_sk','cs_ship_addr_sk','cs_promo_sk','cs_item_sk','cs_quantity',
    'cs_sold_date_sk','cs_sold_time_sk','cs_ship_date_sk','cs_warehouse_sk','ss_sold_date_sk','ss_sold_time_sk','ss_item_sk','ss_customer_sk','ss_cdemo_sk','ss_hdemo_sk','ss_addr_sk','ss_store_sk','ss_promo_sk','ss_quantity'
]

# Change bigint to int and save as delta tables
for table in tables:
    df = spark.read.parquet(f"{volume_path}/{table}")
    for col in int_columns:
        if col in df.columns and dict(df.dtypes)[col] == 'bigint':
            df = df.withColumn(col, df[col].cast('int'))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{table}")

In [0]:
# Amend date_dim to bring time forward and delete unused years
# Add d_date_sk_1 as the first column and overwrite the table
df = spark.table(f"{catalog_name}.{schema_name}.date_dim")
df = df.withColumn("d_date_sk_1", df["d_date_sk"] - 8527)
cols = ["d_date_sk_1"] + [col for col in df.columns if col != "d_date_sk_1"]
df = df.select(cols)
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{schema_name}.date_dim")

# Delete rows where d_date is < "2021-01-01" or > "2026-12-31"
spark.sql(f"""
DELETE FROM {catalog_name}.{schema_name}.date_dim
WHERE d_date < '2021-01-01' OR d_date > '2026-12-31'
""")

In [0]:
# Delete Nulls to ensure Referential Integrity on Facts
# Delete Nulls from catalog_sales
spark.sql(f"""
DELETE FROM {catalog_name}.{schema_name}.catalog_sales
WHERE {' OR '.join([f'cs_{col} IS NULL' for col in ['sold_date_sk','sold_time_sk','ship_date_sk','bill_customer_sk','bill_cdemo_sk','bill_hdemo_sk','bill_addr_sk','ship_customer_sk','ship_cdemo_sk','ship_hdemo_sk','ship_addr_sk','call_center_sk','catalog_page_sk','ship_mode_sk','warehouse_sk','item_sk','promo_sk','order_number','quantity','wholesale_cost','list_price','sales_price','ext_discount_amt','ext_sales_price','ext_wholesale_cost','ext_list_price','ext_tax','coupon_amt','ext_ship_cost','net_paid','net_paid_inc_tax','net_paid_inc_ship','net_paid_inc_ship_tax','net_profit']])}
""")

# Delete Nulls from store_sales
spark.sql(f"""
DELETE FROM {catalog_name}.{schema_name}.store_sales
WHERE {' OR '.join([f'ss_{col} IS NULL' for col in ['sold_date_sk','sold_time_sk','item_sk','customer_sk','cdemo_sk','hdemo_sk','addr_sk','store_sk','promo_sk','ticket_number','quantity','wholesale_cost','list_price','sales_price','ext_discount_amt','ext_sales_price','ext_wholesale_cost','ext_list_price','ext_tax','coupon_amt','net_paid','net_paid_inc_tax','net_profit']])}
""")

In [0]:
# Add a cache_buster column for load testing
for table in ['catalog_sales', 'store_sales']:
    full_table_name = f"{catalog_name}.{schema_name}.{table}"
    spark.sql(f"""
        ALTER TABLE {full_table_name}
        ADD COLUMN cache_buster INT
    """)
    spark.sql(f"""
        UPDATE {full_table_name}
        SET cache_buster = 1
    """)

In [0]:
# Add Primary Key Constraints and Rely Clause on Dim Tables 
spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.date_dim
ALTER COLUMN d_date_sk_1 SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.date_dim
ADD CONSTRAINT pk_date_dim PRIMARY KEY (d_date_sk_1) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_address
ALTER COLUMN ca_address_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_address
ADD CONSTRAINT pk_customer_address PRIMARY KEY (ca_address_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.promotion
ALTER COLUMN p_promo_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.promotion
ADD CONSTRAINT pk_promotion PRIMARY KEY (p_promo_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.item
ALTER COLUMN i_item_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.item
ADD CONSTRAINT pk_item PRIMARY KEY (i_item_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.catalog_page
ALTER COLUMN cp_catalog_page_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.catalog_page
ADD CONSTRAINT pk_cp PRIMARY KEY (cp_catalog_page_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_demographics
ALTER COLUMN cd_demo_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.customer_demographics
ADD CONSTRAINT pk_cd PRIMARY KEY (cd_demo_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.ship_mode
ALTER COLUMN sm_ship_mode_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.ship_mode
ADD CONSTRAINT pk_sm PRIMARY KEY (sm_ship_mode_sk) RELY
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.store
ALTER COLUMN s_store_sk SET NOT NULL
""")

spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.store
ADD CONSTRAINT pk_s PRIMARY KEY (s_store_sk) RELY
""")

In [0]:
# Liquid cluster store_sales by ss_sold_date_sk, ss_addr_sk
spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.store_sales
CLUSTER BY (ss_sold_date_sk, ss_addr_sk)
""")

# Liquid cluster catalog_sales by ss_sold_date_sk, cs_bill_addr_sk
spark.sql(f"""
ALTER TABLE {catalog_name}.{schema_name}.catalog_sales
CLUSTER BY (cs_sold_date_sk, cs_bill_addr_sk)
""")

In [0]:
# Force reclustering 
spark.sql(f"OPTIMIZE {catalog_name}.{schema_name}.store_sales FULL")
spark.sql(f"OPTIMIZE {catalog_name}.{schema_name}.catalog_sales FULL")

In [0]:
# Run VACUUM and ANALYZE on Fact Tables
fact_tables = ['catalog_sales', 'store_sales']

for fact_table in fact_tables:
    full_fact_table_name = f"{catalog_name}.{schema_name}.{fact_table}"
    spark.sql(f"VACUUM {full_fact_table_name}")
    spark.sql(f"ANALYZE TABLE {full_fact_table_name} COMPUTE STATISTICS FOR ALL COLUMNS")

In [0]:
# Run OPTIMIZE, VACUUM and ANALYZE on Dim Tables
dim_tables = ['catalog_page', 'customer_address', 'customer_demographics', 'date_dim', 'item', 'promotion', 'ship_mode', 'store']

for dim_table in dim_tables:
    full_dim_table_name = f"{catalog_name}.{schema_name}.{dim_table}"
    spark.sql(f"OPTIMIZE {full_dim_table_name}")
    spark.sql(f"VACUUM {full_dim_table_name}")
    spark.sql(f"ANALYZE TABLE {full_dim_table_name} COMPUTE STATISTICS FOR ALL COLUMNS")